### Load Files

In [1]:
import pandas as pd

A = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\21-22\ASI_DATA_2021_22_CSV\blkA202122.csv")
B = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\21-22\ASI_DATA_2021_22_CSV\blkB202122.csv")
E = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\21-22\ASI_DATA_2021_22_CSV\blkE202122.csv")
F = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\21-22\ASI_DATA_2021_22_CSV\blkF202122.csv")
G = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\21-22\ASI_DATA_2021_22_CSV\blkG202122.csv")
J = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\21-22\ASI_DATA_2021_22_CSV\blkJ202122.csv")

### Inspect Blocks

In [2]:
for name, df in {
    "A":A,
    "B":B,
    "E":E,
    "F":F,
    "G":G,
    "J":J
}.items():

    print("\n"+"="*80)
    print(f"BLOCK {name}")
    print("="*80)

    print("Shape:",df.shape)
    print("Columns:",df.columns.tolist())
    print(df.head())


BLOCK A
Shape: (68548, 22)
Columns: ['yr', 'blk', 'a1', 'a2', 'a3', 'a4', 'a5', 'a7', 'a8', 'a9', 'a10', 'a11', 'a12', 'bonus', 'pf', 'welfare', 'mwdays', 'nwdays', 'wdays', 'costop', 'expshare', 'mult']
   yr blk      a1     a2  a3    a4     a5  a7  a8  a9  ...  a12  bonus   pf  \
0  22   A  100001  99999   1  9999  12007   1  99   2  ...    2    0.0  0.0   
1  22   A  100002  99999   1  9999  12007   1  99   2  ...    1    0.0  0.0   
2  22   A  100003  99999   1  9999  12007   1  99   2  ...    3    0.0  0.0   
3  22   A  100004  99999   1  9999  12007   1  99   2  ...    1    0.0  0.0   
4  22   A  100005  99999   1  9999  12007   1  99   2  ...    1    0.0  0.0   

    welfare  mwdays  nwdays  wdays      costop  expshare  mult  
0       0.0       0     365    365    256904.0         0   1.0  
1       0.0     305       0    305  23458660.0         0   1.0  
2       0.0       0       0      0         0.0         0   1.0  
3       0.0     301       0    301   5988298.0         0   1

### Define CBAM NIC Codes AND Filter CBAM Firms

In [3]:
cbam_codes = [
    24101,24102,24103,24104,24105,24106,24107,
    24201,24202,24203,24204,24209,
    23941,23942,
    20121,20122,20129,20130
]
cbam = A[A["a5"].isin(cbam_codes)]

print(cbam.shape)
print(cbam["a5"].value_counts())

(2366, 22)
a5
24105    404
24103    261
24202    241
23942    187
24209    174
24102    144
20129    137
23941    123
24101    117
24104    106
24201    105
24106     92
24203     90
20121     84
20122     62
24204     22
24107     17
Name: count, dtype: int64


### Keep Exporters Only

In [4]:
cbam_exporters = cbam[
    cbam["expshare"] > 0
]

print(cbam_exporters.shape)

print(
    cbam_exporters["expshare"].describe()
)

print(
    cbam_exporters[
        ["a1","a5","expshare"]
    ].sort_values(
        "expshare",
        ascending=False
    ).head(20)
)

(150, 22)
count    150.000000
mean      25.133333
std       27.462156
min        1.000000
25%        7.000000
50%       12.000000
75%       33.750000
max      100.000000
Name: expshare, dtype: float64
           a1     a5  expshare
32199  137493  24104       100
32499  137821  24104       100
40856  151179  24202        99
32208  137502  24104        95
35306  142088  20121        95
34249  140370  24106        94
28818  133181  24209        93
18601  120734  24201        93
19281  121525  24104        92
34214  140323  24209        91
28726  133062  24106        88
41358  151959  24201        85
24307  127309  24105        80
19354  121608  24202        80
28699  133026  24105        75
34160  140255  24106        74
32214  137509  24202        67
19265  121503  24104        62
44073  156044  24209        62
40751  150985  24203        62


### Employment Aggregation

In [5]:
emp = (
    E.groupby("AE01")
     .sum(numeric_only=True)
     .reset_index()
)

print(emp.head())

     AE01   yr  E11    E13  E14    E15  E16    E17        E18
0  100001  198   45      0  730    730    2    730   312000.0
1  100002  198   45  12040    0  12040   40  14256  3616086.0
2  100004  198   45   1566    0   1566    6   1936   665520.0
3  100005  198   45  16156    0  16156   58  17502  6980784.0
4  100006  198   45      0  604    604    2    730   312000.0


### Output Aggregation

In [6]:
output = (
    J.groupby("AJ01")["J113"]
     .sum()
     .reset_index()
)

output.columns = [
    "factory",
    "output_value"
]

print(output.head())

   factory  output_value
0   100002    53922128.0
1   100004    12103220.0
2   100005    59582548.0
3   100007    31085228.0
4   100008    20982700.0


### Merge Employment AND OUTPUT

In [7]:
master = cbam_exporters.merge(
    emp,
    left_on="a1",
    right_on="AE01",
    how="left"
)
master = master.merge(
    output,
    left_on="a1",
    right_on="factory",
    how="left"
)

print(master.shape)
print(master.head())

(150, 33)
   yr_x blk      a1     a2  a3    a4     a5  a7  a8  a9  ...  yr_y  E11  \
0    22   A  100273  99999   1  9999  24107   1  99   2  ...   198   45   
1    22   A  101997  99999   1  9999  24103   3  99   2  ...   198   45   
2    22   A  102004  99999   1  9999  24105   3  99   2  ...   198   45   
3    22   A  102013  99999   1  9999  24105   3  99   2  ...   198   45   
4    22   A  102024  99999   1  9999  24105   3  99   2  ...   198   45   

       E13  E14      E15   E16      E17           E18  factory  output_value  
0    57363    0    57363   188    66507  3.364924e+07   100273  2.844561e+08  
1  1319022    0  1319022  4256  1565928  1.137783e+09   101997  2.082902e+10  
2  2006012    0  2006012  6471  2262086  1.795072e+09   102004  2.834459e+10  
3   300860    0   300860   826   372044  2.232320e+08   102013  5.364391e+09  
4   392162    0   392162  1286   461252  2.477720e+08   102024  7.752634e+09  

[5 rows x 33 columns]


### Check Missing Values

In [8]:
print(
    master[
        ["E13","E15","E18","output_value"]
    ].isna().sum()
)

E13             0
E15             0
E18             0
output_value    0
dtype: int64


### Save Excel

In [11]:
master.to_excel(
    "CBAM_Exporters_2021_22.xlsx",
    index=False
)

### Final Validation

In [9]:
print("CBAM firms:", cbam.shape)

print("Exporting CBAM firms:",
      cbam_exporters.shape)

print("Final dataset:",
      master.shape)

print(
    master["expshare"].describe()
)

CBAM firms: (2366, 22)
Exporting CBAM firms: (150, 22)
Final dataset: (150, 33)
count    150.000000
mean      25.133333
std       27.462156
min        1.000000
25%        7.000000
50%       12.000000
75%       33.750000
max      100.000000
Name: expshare, dtype: float64


### Verify whether you have Block I

In [1]:
import pandas as pd

I = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\21-22\ASI_DATA_2021_22_CSV\blkI202122.csv")
print(I.columns.tolist())
print(I.head())

['yr', 'blk', 'AI01', 'I11', 'I13', 'I14', 'I15', 'I16', 'I17']
   yr blk    AI01  I11      I13  I14          I15           I16        I17
0  22   I  100065    1   164000    9  21893075.42  4.162093e+09     190.11
1  22   I  100065    7  9994000    0         0.00  4.162093e+09       0.00
2  22   I  100068    1   137600   27      1090.00  3.521466e+08  323070.29
3  22   I  100068    2   137900   27      1107.00  6.542357e+08  590998.86
4  22   I  100068    7  9994000    0         0.00  1.006382e+09       0.00


In [2]:
print(sorted(I["I13"].unique())[:100])

[np.int64(112100), np.int64(112200), np.int64(115200), np.int64(117100), np.int64(117200), np.int64(119000), np.int64(121300), np.int64(122100), np.int64(123100), np.int64(123200), np.int64(123500), np.int64(124900), np.int64(125100), np.int64(125300), np.int64(125400), np.int64(125900), np.int64(126000), np.int64(129003), np.int64(129099), np.int64(131400), np.int64(131800), np.int64(132900), np.int64(134900), np.int64(135300), np.int64(135900), np.int64(136000), np.int64(137100), np.int64(137200), np.int64(137400), np.int64(137500), np.int64(137600), np.int64(137900), np.int64(139100), np.int64(141200), np.int64(144100), np.int64(144400), np.int64(144500), np.int64(144901), np.int64(144999), np.int64(145000), np.int64(146000), np.int64(149100), np.int64(161001), np.int64(162002), np.int64(162099), np.int64(164000), np.int64(165100), np.int64(165200), np.int64(165300), np.int64(165400), np.int64(165500), np.int64(165700), np.int64(165900), np.int64(169000), np.int64(170101), np.int64(

In [3]:
fuel21 = (
    I.groupby("I13")["I16"]
    .sum()
    .sort_values(ascending=False)
)

fuel21.head(30)

I13
9994000    1.977241e+13
1201000    6.438698e+12
1639001    2.142002e+12
1101002    1.245562e+12
9922100    5.966124e+11
2153500    5.096504e+11
1202003    3.939697e+11
1632001    3.275651e+11
3337000    3.059621e+11
2153100    2.866664e+11
1421000    2.536527e+11
4912999    2.191893e+11
3411071    1.880129e+11
1424004    1.523057e+11
4740100    1.479904e+11
3423200    1.421457e+11
4740300    1.369228e+11
2153300    1.354468e+11
1424099    9.782906e+10
1101006    9.639825e+10
3936102    9.520774e+10
4714099    8.760020e+10
1639006    8.590210e+10
4715002    7.982444e+10
3934003    7.560257e+10
4121101    6.831147e+10
3935001    6.469154e+10
3924001    6.263967e+10
3338007    5.875520e+10
1631002    5.626613e+10
Name: I16, dtype: float64